# Z-Score Bollinger Strategy on SPY
## Strategy Brief
The Z-Score Bollinger Strategy is a mean-reversion trading strategy that uses Bollinger Bands and Z-Score to identify overbought and oversold conditions in SPY, an ETF that tracks the S&P 500 index. The strategy generates a buy signal when the price crosses below the lower Bollinger Band and the Z-Score indicates oversold conditions, and a sell signal when the price crosses above the upper Bollinger Band and the Z-Score indicates overbought conditions. The goal is to capture price reversals back towards the mean. Historical backtesting of this strategy on SPY data aims to assess its profitability and risk-adjusted returns.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
In this phase, we define the parameters for the Z-Score Bollinger Strategy, including the lookback period for the Bollinger Bands, the number of standard deviations for the bands, and the lookback period for the Z-Score calculation.

In [ ]:
LOOKBACK_PERIOD = 20
STD_DEV = 2
ZSCORE_LOOKBACK = 20
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'

## PHASE 2 - Data Exploration
We will download historical SPY data using yfinance and compute the Bollinger Bands and Z-Score indicators. These indicators will be plotted over the SPY price to visualize potential trading signals.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download('SPY', start=START_DATE, end=END_DATE)

# Calculate Bollinger Bands
rolling_mean = data['Close'].rolling(window=LOOKBACK_PERIOD).mean()
rolling_std = data['Close'].rolling(window=LOOKBACK_PERIOD).std()
data['Upper Band'] = rolling_mean + (rolling_std * STD_DEV)
data['Lower Band'] = rolling_mean - (rolling_std * STD_DEV)

# Calculate Z-Score
rolling_mean_z = data['Close'].rolling(window=ZSCORE_LOOKBACK).mean()
rolling_std_z = data['Close'].rolling(window=ZSCORE_LOOKBACK).std()
data['Z-Score'] = (data['Close'] - rolling_mean_z) / rolling_std_z

# Plot
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='SPY Close')
plt.plot(data['Upper Band'], label='Upper Bollinger Band', linestyle='--')
plt.plot(data['Lower Band'], label='Lower Bollinger Band', linestyle='--')
plt.title('SPY Price with Bollinger Bands')
plt.legend()
plt.show()

## PHASE 3 - Strategy Engineering
In this phase, we define the trading signals based on the Z-Score and Bollinger Bands. A buy signal is generated when the price crosses below the lower band and the Z-Score is below -1. A sell signal is generated when the price crosses above the upper band and the Z-Score is above 1.

In [ ]:
# Generate trading signals
data['Signal'] = 0
# Buy signal
buy_signal = (data['Close'] < data['Lower Band']) & (data['Z-Score'] < -1)
data.loc[buy_signal, 'Signal'] = 1
# Sell signal
sell_signal = (data['Close'] > data['Upper Band']) & (data['Z-Score'] > 1)
data.loc[sell_signal, 'Signal'] = -1

# Generate positions
data['Position'] = data['Signal'].replace(to_replace=0, method='ffill')

## PHASE 4 - Coding & Backtesting
We will simulate the trading strategy by calculating daily returns based on the positions and plot the resulting equity curve to visualize the strategy's performance over time.

In [ ]:
# Calculate daily returns
data['Market Return'] = data['Close'].pct_change()
data['Strategy Return'] = data['Position'].shift(1) * data['Market Return']

data['Equity Curve'] = (1 + data['Strategy Return']).cumprod()
data['Equity Curve (Buy and Hold)'] = (1 + data['Market Return']).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data['Equity Curve'], label='Strategy Equity Curve')
plt.plot(data['Equity Curve (Buy and Hold)'], label='Buy and Hold Equity Curve')
plt.title('Equity Curve')
plt.legend()
plt.show()

## PHASE 5 - Performance Evaluation
In this phase, we evaluate the strategy's performance using metrics such as CAGR, Sharpe Ratio, Sortino Ratio, Calmar Ratio, and Maximum Drawdown. We compare these metrics against a simple buy-and-hold strategy.

In [ ]:
def calculate_performance_metrics(data):
    # CAGR
    years = (data.index[-1] - data.index[0]).days / 365.25
    cagr = (data['Equity Curve'].iloc[-1]) ** (1 / years) - 1
    
    # Sharpe Ratio
    sharpe_ratio = data['Strategy Return'].mean() / data['Strategy Return'].std() * np.sqrt(252)
    
    # Sortino Ratio
    downside_std = data[data['Strategy Return'] < 0]['Strategy Return'].std()
    sortino_ratio = data['Strategy Return'].mean() / downside_std * np.sqrt(252)
    
    # Calmar Ratio
    max_drawdown = (data['Equity Curve'].cummax() - data['Equity Curve']).max()
    calmar_ratio = cagr / max_drawdown
    
    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

strategy_metrics = calculate_performance_metrics(data)
buy_and_hold_metrics = calculate_performance_metrics(data[['Equity Curve (Buy and Hold)']].rename(columns={'Equity Curve (Buy and Hold)': 'Equity Curve'}))

comparison_table = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Strategy': strategy_metrics,
    'Buy and Hold': buy_and_hold_metrics
})

print(comparison_table)

## PHASE 6 - Deploy & Monitor
Finally, we create a function to download the last 60 days of SPY data, compute today's signals, and print whether to hold a long or short position based on the strategy.

In [ ]:
def get_latest_signal():
    # Download last 60 days of data
    recent_data = yf.download('SPY', period='60d')
    
    # Calculate indicators
    rolling_mean = recent_data['Close'].rolling(window=LOOKBACK_PERIOD).mean()
    rolling_std = recent_data['Close'].rolling(window=LOOKBACK_PERIOD).std()
    recent_data['Upper Band'] = rolling_mean + (rolling_std * STD_DEV)
    recent_data['Lower Band'] = rolling_mean - (rolling_std * STD_DEV)
    
    rolling_mean_z = recent_data['Close'].rolling(window=ZSCORE_LOOKBACK).mean()
    rolling_std_z = recent_data['Close'].rolling(window=ZSCORE_LOOKBACK).std()
    recent_data['Z-Score'] = (recent_data['Close'] - rolling_mean_z) / rolling_std_z
    
    # Determine latest signal
    latest_row = recent_data.iloc[-1]
    if latest_row['Close'] < latest_row['Lower Band'] and latest_row['Z-Score'] < -1:
        print('Buy Signal')
    elif latest_row['Close'] > latest_row['Upper Band'] and latest_row['Z-Score'] > 1:
        print('Sell Signal')
    else:
        print('Hold')

get_latest_signal()